## IEC Voter Registration Extraction Approach

Before extracting the data, the IEC Voter Registration Statistics page was inspected to understand how the website delivers its data.

The page does not expose the required municipality and ward registration data directly in the initial HTML. The province, municipality, and ward selections are handled through the IEC's ASP.NET Web Forms interface.

The browser's network activity was therefore inspected to identify what happens when selections are made. This showed that selecting a province triggers an asynchronous POST request to the same IEC page, carrying ASP.NET form-state fields such as `__VIEWSTATE`, `__EVENTVALIDATION`, the selected province, and the event target.

Based on this observation, we will reproduce the website's normal selection process programmatically rather than trying to invent or rely on an undocumented API.

### Extraction flow

`IEC page → KwaZulu-Natal → Municipality → Ward → Registration statistics`

The extraction will:

1. Start an HTTP session with the IEC website.
2. Load the initial page and obtain the required ASP.NET state fields.
3. Select **KwaZulu-Natal** programmatically.
4. Retrieve the available KwaZulu-Natal municipalities.
5. Select each municipality and retrieve its wards.
6. Retrieve the registration statistics available for each ward.
7. Repeat this across the required **2011–2026** period where the IEC source provides the corresponding data.
8. Verify the extracted structure and records.
9. Save the original extracted results to `data/raw/`.

This notebook is limited to **data extraction and verification**. Cleaning, transformation, feature engineering, analysis, and modelling will be handled later in the project.

In [1]:
# We are extracting voter registration statistics
# for KwaZulu-Natal only. from2011 to 2026
#
# The IEC website uses an ASP.NET form, so we will
# reproduce the same province → municipality → ward
# selection process programmatically.
#
# Raw extracted data will be saved in:
# data/raw/
#
# No cleaning or processing is done in this notebook.

from pathlib import Path
import pandas as pd
import requests
from bs4 import BeautifulSoup

# Project root
PROJECT_ROOT = Path.cwd().parent

# Raw data directory
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# IEC source
IEC_URL = (
    "https://www.elections.org.za/pw/StatsData/"
    "Voter-Registration-Statistics"
)

# Extraction scope
TARGET_PROVINCE = "KwaZulu-Natal"
KZN_PROVINCE_ID = "4"

# Historical period requested
START_YEAR = 2011
CURRENT_YEAR = 2026

print("IEC extraction setup complete.")
print("Source:", IEC_URL)
print("Province:", TARGET_PROVINCE)
print("Province ID:", KZN_PROVINCE_ID)
print("Period:", START_YEAR, "to", CURRENT_YEAR)
print("Raw directory:", RAW_DIR)

IEC extraction setup complete.
Source: https://www.elections.org.za/pw/StatsData/Voter-Registration-Statistics
Province: KwaZulu-Natal
Province ID: 4
Period: 2011 to 2026
Raw directory: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\raw


In [2]:
#2.
# We first open the IEC page and keep a session active.
# The session is needed because the IEC uses ASP.NET
# state and cookies when moving between selections.

iec_session = requests.Session()

initial_response = iec_session.get(
    IEC_URL,
    timeout=30
)

print("HTTP status:", initial_response.status_code)
print("Content type:", initial_response.headers.get("Content-Type"))
print("Response size:", len(initial_response.content), "bytes")

assert initial_response.status_code == 200, (
    "IEC voter registration page could not be loaded."
)

initial_soup = BeautifulSoup(
    initial_response.text,
    "html.parser"
)

print("IEC source loaded successfully.")
print("Session established successfully.")

HTTP status: 200
Content type: text/html; charset=utf-8
Response size: 131362 bytes
IEC source loaded successfully.
Session established successfully.


In [3]:
#3.
# The IEC uses ASP.NET Web Forms.
# We need the hidden state fields generated by
# the page before we can reproduce the browser's
# province-selection request.
#
# These values are extracted dynamically because
# they can change between sessions.

form_state = {}

for field in initial_soup.select(
    "input[type='hidden'][name]"
):
    name = field.get("name")
    value = field.get("value", "")

    form_state[name] = value

print("Hidden form fields found:", len(form_state))

required_fields = [
    "__VIEWSTATE",
    "__VIEWSTATEGENERATOR",
    "__EVENTVALIDATION"
]

for field in required_fields:
    assert field in form_state, (
        f"Required ASP.NET field missing: {field}"
    )

print("Required ASP.NET form state found.")

print("\nRequired fields:")
for field in required_fields:
    print("-", field)

Hidden form fields found: 3
Required ASP.NET form state found.

Required fields:
- __VIEWSTATE
- __VIEWSTATEGENERATOR
- __EVENTVALIDATION


In [6]:
#4.
# Lets select the province we want to focus on
# The IEC uses ASP.NET AJAX for the province dropdown.
# Our earlier request reached the server but returned
# ER-500, so we now reproduce the browser request more
# closely, including the AJAX headers.

province_post_data = form_state.copy()

province_post_data.update({
    "ctl00$ctl13":
        "ctl00$MainContent$MainUpdatePanel|"
        "ctl00$MainContent$ddlProvinces",

    "__EVENTTARGET":
        "ctl00$MainContent$ddlProvinces",

    "__EVENTARGUMENT":
        "",

    "__LASTFOCUS":
        "",

    "__SCROLLPOSITIONX":
        "0",

    "__SCROLLPOSITIONY":
        "0",

    "ctl00$MainContent$ddlProvinces":
        KZN_PROVINCE_ID,

    "ctl00$MainContent$ddlMunicipalities":
        "-1",

    "__ASYNCPOST":
        "true"
})

province_headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/153.0.0.0 Safari/537.36"
    ),
    "Accept": "*/*",
    "X-MicrosoftAjax": "Delta=true",
    "X-Requested-With": "XMLHttpRequest",
    "Referer": IEC_URL,
    "Origin": "https://www.elections.org.za"
}

province_response = iec_session.post(
    IEC_URL,
    data=province_post_data,
    headers=province_headers,
    timeout=30
)

print("HTTP status:", province_response.status_code)
print("Content type:", province_response.headers.get("Content-Type"))
print("Response size:", len(province_response.content), "bytes")

print("\nResponse preview:")
print(province_response.text[:300])

HTTP status: 200
Content type: text/plain; charset=utf-8
Response size: 129533 bytes

Response preview:
1|#||4|112264|updatePanel|MainContent_MainUpdatePanel|



            <div class="container-fluid">



                <div class="blue_background">
                    <div class="col p-0 m-0 subpage_banner">



                        <div class="card-deck subbanner h-100" style="padd


In [9]:
#5. 
# ============================================
# CELL 5 — VERIFY KZN PROVINCE RESPONSE
# ============================================
#
# The IEC returned an ASP.NET AJAX updatePanel response.
# We extract the updated HTML and check that the
# municipality dropdown was populated for KwaZulu-Natal.

response_text = province_response.text

assert "updatePanel|MainContent_MainUpdatePanel|" in response_text, (
    "IEC did not return the expected updatePanel response."
)

# Extract the HTML contained in the updatePanel response
panel_marker = "updatePanel|MainContent_MainUpdatePanel|"

panel_start = response_text.find(panel_marker) + len(panel_marker)
panel_html = response_text[panel_start:]

# Parse the returned HTML
province_soup = BeautifulSoup(
    panel_html,
    "html.parser"
)

# Find the municipality dropdown
municipality_select = province_soup.select_one(
    "select[name='ctl00$MainContent$ddlMunicipalities']"
)

assert municipality_select is not None, (
    "Municipality dropdown was not found in the KZN response."
)

# Extract municipality options
municipality_options = []

for option in municipality_select.find_all("option"):
    value = option.get("value", "").strip()
    text = option.get_text(strip=True)

    if value and value != "-1":
        municipality_options.append({
            "municipality_id": value,
            "municipality": text
        })

print("KZN province response verified.")
print("Municipalities found:", len(municipality_options))

print("\nFirst municipalities:")
for municipality in municipality_options[:10]:
    print(
        municipality["municipality_id"],
        "→",
        municipality["municipality"]
    )

KZN province response verified.
Municipalities found: 44

First municipalities:
4005 → ETH - eThekwini
4403 → KZN212 - uMdoni
4404 → KZN213 - uMzumbe
4405 → KZN214 - uMuziwabantu
4407 → KZN216 - Ray Nkonyeni
4409 → KZN221 - uMshwathi
4410 → KZN222 - uMngeni
4411 → KZN223 - Mpofana
4412 → KZN224 - iMpendle
4413 → KZN225 - Msunduzi


In [13]:
# ============================================
# CELL 6 — INSPECT SUCCESSFUL KZN RESPONSE
# ============================================
#
# The KZN AJAX request succeeded, but its response
# does not contain the hidden ASP.NET fields.
#
# We inspect the response structure before constructing
# the municipality request.

response_text = province_response.text

print("Response length:", len(response_text))

print("\n--- RESPONSE CONTROL RECORDS ---")

for marker in [
    "updatePanel|",
    "asyncPostBackControlIDs|",
    "postBackControlIDs|",
    "updatePanelIDs|",
    "panelsToRefreshIDs|",
    "formAction|",
    "pageTitle|",
    "scriptBlock|"
]:
    position = response_text.find(marker)

    if position >= 0:
        print(
            f"\n{marker}"
            f"\n{response_text[position:position + 500]}"
        )
    else:
        print(f"\n{marker} NOT FOUND")

Response length: 129533

--- RESPONSE CONTROL RECORDS ---

updatePanel|
updatePanel|MainContent_MainUpdatePanel|



            <div class="container-fluid">



                <div class="blue_background">
                    <div class="col p-0 m-0 subpage_banner">



                        <div class="card-deck subbanner h-100" style="padding-top: 0px !important;">
                            <div class="card-body d-flex flex-column" style="padding-bottom: 0px !important;">
                                <h1 class="card-title white_text display-4

asyncPostBackControlIDs|
asyncPostBackControlIDs|||0|postBackControlIDs|||62|updatePanelIDs||tctl00$MainContent$MainUpdatePanel,MainContent_MainUpdatePanel|0|childUpdatePanelIDs|||61|panelsToRefreshIDs||ctl00$MainContent$MainUpdatePanel,MainContent_MainUpdatePanel|2|asyncPostBackTimeout||90|31|formAction||./Voter-Registration-Statistics|29|pageTitle||Voter Registration Statistics|

postBackControlIDs|
postBackControlIDs|||62|updatePanelI